In [0]:
df_gold = spark.read.table("workspace.default.gold_vuelos_features")

###  Auditoría de Calidad y Validación de Rangos

Esta celda realiza dos controles críticos de calidad de datos:

**1. Detección de valores nulos:**
* Cuenta los valores `NULL` en cada columna usando `isNull()`
* Todas las columnas deben tener 0 nulos para garantizar integridad

**2. Validación de rangos:**
* **Variables cíclicas** (sin/cos): deben estar en el rango [-1, 1]
* **Variables escaladas** (clima): deben estar en el rango [0, 1]
* **Variable `label`**: debe contener únicamente valores {0, 1, 2}

Cualquier desviación en estos rangos indica problemas en la capa Silver que deben corregirse.

In [0]:
# --------------------------------------------------------------
# ML CLÁSICO - CELDA 2: AUDITORÍA DE CALIDAD Y CONTROL DE RANGOS 
# -------------------------------------------------------------------
import pyspark.sql.functions as F

print("--- CONTROL DE CALIDAD SENSORIAL DEL DATASET GOLD ---")

# 1. Conteo de Nulos absolutos adaptativo al tipo de dato
# Usamos isNull() para todo que es el estándar seguro para fechas, strings y números
df_auditoria_nulos = df_gold.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_gold.columns
])

print("\n Conteo de Nulos por columna (Debe ser todo 0):")
df_auditoria_nulos.show()

# 2. Validamos los rangos geométricos y climáticos (Mínimos y Máximos)
print(" Rangos de las variables escaladas y cíclicas:")
df_gold.select(
    "sin_month", "cos_month",
    "scaled_max_temp", "scaled_precipitation", "scaled_wind_speed",
    "label"
).summary("min", "max").show()

###  Sanitización Integral sin Pérdida de Registros

Esta celda aplica tres transformaciones de limpieza para garantizar la calidad del dataset:

**1. Imputación de nulos con 0.0:**
* Reemplaza valores faltantes en variables escaladas y cíclicas
* Estrategia conservadora: el valor 0 representa ausencia de señal

**2. Capeo de variables escaladas al rango [0, 1]:**
* Corrige valores climáticos que exceden el rango normalizado
* Aplica a: `scaled_max_temp`, `scaled_min_temp`, `scaled_precipitation`, `scaled_snowfall`, `scaled_wind_speed`

**3. Capeo de variables cíclicas al rango [-1, 1]:**
* Corrige errores numéricos en transformaciones trigonométricas
* Aplica a: sin/cos de mes, día de semana, día de mes, y variables de volumen

**Resultado:** Dataset completamente limpio sin eliminar registros, preservando los 8.6M de vuelos.

In [0]:
# ----------------------------------------------------------------------------
# ML CLÁSICO - CELDA 3: SANITIZACIÓN INTEGRAL (SIN BORRAR FILAS)
# ----------------------------------------------------------------------------
import pyspark.sql.functions as F
print("--- REPARANDO DATASET MEDIANTE IMPUTACIÓN Y CAPEO DE RANGOS ---")
# Variables escaladas
columnas_escaladas = [
    "scaled_max_temp",
    "scaled_min_temp",
    "scaled_precipitation",
    "scaled_snowfall",
    "scaled_wind_speed"
]

# Variables cíclicas
columnas_ciclicas = [
    "sin_month",
    "cos_month",
    "sin_dayofweek",
    "cos_dayofweek",
    "sin_dayofmonth",
    "cos_dayofmonth",
    "sin_hour",
    "cos_hour"
]

# Variables operacionales continuas 
columnas_operacionales = [
    "CRSElapsedTime",
    "Distance",
    "scaled_origin_volume",
    "scaled_dest_volume"
]

# 1. Imputación de nulos con 0.0
df_gold_sanitizado = df_gold.fillna(
    value=0.0,
    subset=columnas_escaladas + columnas_ciclicas + columnas_operacionales
)

# 2. Capeo de variables escaladas al intervalo [0,1]
for c in columnas_escaladas:
    df_gold_sanitizado = (
        df_gold_sanitizado
        .withColumn(
            c,
            F.when(F.col(c) < 0.0, 0.0)
             .when(F.col(c) > 1.0, 1.0)
             .otherwise(F.col(c))
        )
    )

# 3. Capeo de variables cíclicas al intervalo [-1,1]
for c in columnas_ciclicas:
    df_gold_sanitizado = (
        df_gold_sanitizado
        .withColumn(
            c,
            F.when(F.col(c) < -1.0, -1.0)
             .when(F.col(c) > 1.0, 1.0)
             .otherwise(F.col(c))
        )
    )

# 4. Auditoría final
print("\n--- VERIFICACIÓN FINAL DE RANGOS ---")

df_gold_sanitizado.select(
    "scaled_max_temp",
    "scaled_min_temp",
    "scaled_precipitation",
    "scaled_snowfall",
    "scaled_wind_speed",
    "sin_month",
    "cos_month",
    "sin_hour",
    "cos_hour"
).summary("min", "max").show()

print("\n--- NULOS REMANENTES ---")

df_gold_sanitizado.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in columnas_escaladas + columnas_ciclicas + columnas_operacionales
]).show()

print("\n✓ Dataset sanitizado correctamente.")
print(f"✓ Total de registros preservados: {df_gold_sanitizado.count():,}")

###  Carga del Dataset Gold desde Unity Catalog

Esta celda carga la tabla final procesada `workspace.default.gold_vuelos_features` que contiene:

* **8.6 millones de registros** de vuelos con features engineered
* **Variables cíclicas temporales**: sin/cos de mes, día de semana, día de mes
* **Variables climáticas escaladas**: temperatura, precipitación, nevada, viento (normalizadas [0,1])
* **Variables de volumen escaladas**: tráfico en aeropuertos de origen y destino
* **Variable objetivo (`label`)**: 
  * 0 = Vuelo puntual
  * 1 = Vuelo retrasado (>15 min)
  * 2 = Vuelo cancelado

In [0]:
print("Distribución de clases:")
df_gold_sanitizado.groupBy("label").count().orderBy("label").show()

## Machine Learning


###  Configuración Global y Carga de Librerías

Esta celda establece el ambiente completo de Machine Learning:

**Librerías importadas:**
* **scikit-learn**: Modelos (RandomForest, GradientBoosting, LinearSVC, MLPClassifier, LogisticRegression), validación cruzada, búsqueda de hiperparámetros
* **Métricas**: confusion_matrix, classification_report, accuracy, precision, recall, F1, ROC-AUC
* **Visualización**: matplotlib, seaborn
* **Scipy**: Distribuciones para RandomizedSearchCV
* **LightGBM**: Modelado multicategoría eficiente
* **Joblib**: Serialización de modelos entrenados

**Configuración global:**
* **Semilla SEED=42**: Garantiza reproducibilidad total en todas las operaciones aleatorias
* **Paleta de colores corporativa**: Consistencia visual en todos los gráficos
* **Supresión de warnings**: Limpia la salida de mensajes innecesarios

**Objetivo del notebook:**
Clasificación multiclase de estatus de vuelos (puntual / retrasado / cancelado) usando 8.6M de registros.

In [0]:
# _________________________________________________________________________
# Objetivo:
# Clase 0 → Vuelo puntual
# Clase 1 → Vuelo retrasado (>15 min)
# Clase 2 → Vuelo cancelado
#
# Dataset: 8.6 millones de vuelos
# Seed global: 42
# __________________________________________________________________________

# -----------------------------------------------------------------------------
# LIBRERÍAS CORE
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
import time

# -----------------------------------------------------------------------------
# PARTICIÓN Y VALIDACIÓN
# -----------------------------------------------------------------------------
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score
)

# -----------------------------------------------------------------------------
# MODELOS
# -----------------------------------------------------------------------------
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# -----------------------------------------------------------------------------
# MÉTRICAS
# -----------------------------------------------------------------------------
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# -----------------------------------------------------------------------------
# VISUALIZACIÓN
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# -----------------------------------------------------------------------------
# DISTRIBUCIONES PARA BÚSQUEDA DE HIPERPARÁMETROS
# -----------------------------------------------------------------------------
from scipy.stats import randint, uniform, loguniform

# -----------------------------------------------------------------------------
# CONFIGURACIÓN GLOBAL
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# PALETA CORPORATIVA DEL PROYECTO
# -----------------------------------------------------------------------------
COLORS = {
    "primary": "#1A73E8",
    "danger": "#E84545",
    "success": "#2ECC71",
    "neutral": "#95A5A6",
    "dark": "#1C2833",
    "light_bg": "#F4F6F9"
}

# -----------------------------------------------------------------------------
# INFORMACIÓN DEL PROYECTO
# -----------------------------------------------------------------------------
print("-" * 75)
print(" FLIGHT STATUS MULTICLASS CLASSIFIER")
print("-" * 75)

print("\nClases del problema:")
print("  Clase 0 → Vuelo puntual")
print("  Clase 1 → Retraso superior a 15 minutos")
print("  Clase 2 → Vuelo cancelado")

print(f"\nSeed global fijado en: {SEED}")
print("Librerías cargadas correctamente.")
print("=" * 75)


###  Extracción de Muestra Balanceada desde Spark y Feature Engineering Inicial

Esta celda convierte el dataset distribuido de Spark a Pandas con estrategia de balanceo y agrega primeras interacciones:

**Estrategia de muestreo:**
* **150,000 registros por clase** (450,000 total)
* **Muestreo aleatorio estratificado** con `orderBy(F.rand(seed))` para cada clase
* **Conversión Spark → Pandas** para compatibilidad con scikit-learn
* **Shuffle final** con `sample(frac=1)` para mezclar las tres clases

**Columnas seleccionadas:**
* Variables de contexto: `FlightDate`, `ORIGIN`, `DEST`
* 6 variables cíclicas: sin/cos de mes, día de semana, día de mes
* 5 variables climáticas escaladas: temperatura, precipitación, nevada, viento
* Variables de volumen: `scaled_origin_volume`, `scaled_dest_volume`
* Tiempo y distancia: `CRSElapsedTime`, `Distance`
* Variables cíclicas de hora: `sin_hour`, `cos_hour`
* Target: `label`
* **Aerolínea:** `Reporting_Airline` (one-hot encoding y carrier_id para target encoding)

**Interacciones agregadas:**
* `weather_traffic_origin`: interacción clima × tráfico en origen
* `weather_traffic_dest`: interacción clima × tráfico en destino
* `wind_distance`: viento × distancia

**Validaciones incluidas:**
* Shape esperado: (450000, columnas dinámicas según cantidad de aerolíneas)
* Sin valores nulos
* Las tres clases presentes
* Columnas one-hot de aerolínea generadas

La función es **idempotente**: si `df_ml_pandas` ya existe, no se vuelve a ejecutar.

In [0]:
# ____________________________________________________________________________
# CELDA 4 : PySpark Distribuido -> Pandas Local
# CAMBIO:
# - Usa df_gold_sanitizado y no df_gold directo.
# - Agrega Reporting_Airline.
# - Convierte aerolínea a variables one-hot.
# - Mantiene carrier_id para target encoding posterior.
# ___________________________________________________________________________

def build_multiclass_pandas_df(seed=SEED):

    from pyspark.sql import functions as F

    TARGET_COL = "label"
    N_PER_CLASS = 150_000

    # Se usa la versión sanitizada.
    df_source = df_gold_sanitizado

    COLS_TO_KEEP = [
        "FlightDate",
        "ORIGIN",
        "DEST",
        "Reporting_Airline",
        "sin_month",
        "cos_month",
        "sin_dayofweek",
        "cos_dayofweek",
        "sin_dayofmonth",
        "cos_dayofmonth",
        "scaled_max_temp",
        "scaled_min_temp",
        "scaled_precipitation",
        "scaled_snowfall",
        "scaled_wind_speed",
        "CRSElapsedTime",
        "Distance",
        "scaled_origin_volume",
        "scaled_dest_volume",
        "sin_hour",
        "cos_hour",
        "label"
    ]

    print("=" * 70)
    print("EXTRAYENDO MUESTRA BALANCEADA DESDE SPARK")
    print("=" * 70)

    print(f"Filas disponibles en Gold sanitizado: {df_source.count():,}")

    # Clase 0: puntual
    df_class_0 = (
        df_source
        .filter(F.col(TARGET_COL) == 0.0)
        .select(COLS_TO_KEEP)
        .dropna(subset=COLS_TO_KEEP)
        .orderBy(F.rand(seed))
        .limit(N_PER_CLASS)
    )

    # Clase 1: retrasado
    df_class_1 = (
        df_source
        .filter(F.col(TARGET_COL) == 1.0)
        .select(COLS_TO_KEEP)
        .dropna(subset=COLS_TO_KEEP)
        .orderBy(F.rand(seed))
        .limit(N_PER_CLASS)
    )

    # Clase 2: cancelado
    df_class_2 = (
        df_source
        .filter(F.col(TARGET_COL) == 2.0)
        .select(COLS_TO_KEEP)
        .dropna(subset=COLS_TO_KEEP)
        .orderBy(F.rand(seed))
        .limit(N_PER_CLASS)
    )

    df_balanced_spark = (
        df_class_0
        .union(df_class_1)
        .union(df_class_2)
    )

    print("\nConversión Spark a Pandas...")

    df_ml_pandas = (
        df_balanced_spark
        .toPandas()
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # NUEVO 1: Se guarda una copia de la aerolínea para target encoding.
    # Luego Reporting_Airline se convierte a columnas one-hot.
    # -------------------------------------------------------------------------

    df_ml_pandas["carrier_id"] = (
        df_ml_pandas["Reporting_Airline"]
        .astype(str)
    )

    df_ml_pandas = pd.get_dummies(
        df_ml_pandas,
        columns=["Reporting_Airline"],
        prefix="carrier",
        dtype=np.float32
    )

    # -------------------------------------------------------------------------
    # Interacciones existentes de clima y congestión.
    # -------------------------------------------------------------------------

    print("Calculando interacciones operacionales clima x tráfico...")

    df_ml_pandas["weather_traffic_origin"] = (
        df_ml_pandas["scaled_precipitation"] *
        df_ml_pandas["scaled_origin_volume"] +

        df_ml_pandas["scaled_snowfall"] *
        df_ml_pandas["scaled_origin_volume"] *
        1.5
    )

    df_ml_pandas["weather_traffic_dest"] = (
        df_ml_pandas["scaled_precipitation"] *
        df_ml_pandas["scaled_dest_volume"] +

        df_ml_pandas["scaled_wind_speed"] *
        df_ml_pandas["scaled_dest_volume"]
    )

    max_distance = df_ml_pandas["Distance"].max()

    if max_distance > 0:
        df_ml_pandas["wind_distance"] = (
            df_ml_pandas["scaled_wind_speed"] *
            (df_ml_pandas["Distance"] / max_distance)
        )
    else:
        df_ml_pandas["wind_distance"] = 0.0

    print("\nDataset listo para ML.")
    print(f"Shape final: {df_ml_pandas.shape}")

    return df_ml_pandas


# -----------------------------------------------------------------------------
# EJECUCIÓN
# -----------------------------------------------------------------------------

if "df_ml_pandas" in globals():
    del df_ml_pandas

df_ml_pandas = build_multiclass_pandas_df()

# -----------------------------------------------------------------------------
# VALIDACIONES
# -----------------------------------------------------------------------------

assert df_ml_pandas.shape[0] == 450_000, (
    f"Cantidad inesperada de filas: {df_ml_pandas.shape[0]}"
)

assert df_ml_pandas.isnull().sum().sum() == 0, (
    "Existen nulos residuales"
)

assert set(df_ml_pandas["label"].unique()) == {0.0, 1.0, 2.0}, (
    "No están presentes las tres clases"
)

carrier_cols = [
    c for c in df_ml_pandas.columns
    if c.startswith("carrier_")
]

assert len(carrier_cols) > 0, (
    "No se generaron columnas one-hot de aerolínea"
)

print("\nVALIDACIÓN EXITOSA")
print("=" * 50)
print(df_ml_pandas["label"].value_counts().sort_index())
print(f"\nCantidad de variables one-hot de aerolínea: {len(carrier_cols)}")

###  Feature Engineering y Partición Train/Test

Esta celda realiza el procesamiento final de features y la partición para Machine Learning:

**1. Reconstrucción de hora y ventanas pico:**
* Calcula `hour_approx` a partir de las variables seno/coseno de hora.
* Crea flags binarios: `is_morning_rush` (6-9h) y `is_evening_rush` (16-19h).

**2. Identificador de ruta y aerolínea:**
* Genera `route_id` (`ORIGIN_DEST`) y `carrier_id` para codificación bayesiana de historial de retrasos y cancelaciones.

**3. Selección de features:**
* **Excluidas:** `FlightDate`, `ORIGIN`, `DEST`, `label`, `route_id`, `carrier_id`, `hour_approx`.
* **Incluidas:** Todas las variables numéricas transformadas, flags de hora pico, interacciones clima × tráfico, one-hot de aerolínea, etc.

**4. Validaciones estrictas:**
* Sin valores `NaN` ni infinitos en X.

**5. Train/Test Split (80/20):**
* Estratificado por clase (`stratify=y`), semilla fija (`random_state=SEED`).

**6. Target encoding bayesiano (sin fugas):**
* Inyecta variables `route_hist_delay_rate`, `carrier_hist_delay_rate`, `route_hist_cancel_rate`, `carrier_hist_cancel_rate` usando historial de retrasos/cancelaciones por ruta y aerolínea, calculados solo con datos de entrenamiento (anti-leakage, OOF en train).

El dataset queda listo para entrenar modelos multiclase robustos, con ingeniería de features avanzada y sin fugas de información.

In [0]:
# ______________________________________________________________________________
# CELDA 5 | FEATURE ENGINEERING
# CAMBIO 
# - One-hot de aerolíneas ya creado en la celda anterior.
# - Target encoding separado para retrasos y cancelaciones.
# - OOF target encoding en train para evitar que cada fila use su propia etiqueta.
# _____________________________________________________________________________

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

# -----------------------------------------------------------------------------
# STEP 1: Hora aproximada y ruta
# -----------------------------------------------------------------------------

df_ml_pandas["hour_approx"] = (
    np.arctan2(
        df_ml_pandas["sin_hour"],
        df_ml_pandas["cos_hour"]
    ) * 24 / (2 * np.pi)
) % 24

df_ml_pandas["is_morning_rush"] = (
    (df_ml_pandas["hour_approx"] >= 6) &
    (df_ml_pandas["hour_approx"] <= 9)
).astype(int)

df_ml_pandas["is_evening_rush"] = (
    (df_ml_pandas["hour_approx"] >= 16) &
    (df_ml_pandas["hour_approx"] <= 19)
).astype(int)

df_ml_pandas["route_id"] = (
    df_ml_pandas["ORIGIN"].astype(str) +
    "_" +
    df_ml_pandas["DEST"].astype(str)
)

# -----------------------------------------------------------------------------
# Columnas que no entran directamente a X.
# route_id y carrier_id se usarán para target encoding, pero no como strings.
# -----------------------------------------------------------------------------

COLS_EXCLUDE = [
    "FlightDate",
    "ORIGIN",
    "DEST",
    "label",
    "route_id",
    "carrier_id",
    "hour_approx"
]
FEATURE_COLS = [
    c for c in df_ml_pandas.columns
    if c not in COLS_EXCLUDE
]
X = df_ml_pandas[FEATURE_COLS].copy()
y = df_ml_pandas["label"].astype(int)
print("=" * 70)
print("SEPARACIÓN FEATURES / TARGET")
print("=" * 70)
print(f"\nNúmero de variables predictoras iniciales: {len(FEATURE_COLS)}")
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)
print("\nDistribución de clases:")
print(y.value_counts().sort_index())

assert not X.isnull().any().any(), (
    "[ERROR] Existen NaN en X"
)

assert not np.isinf(X.values).any(), (
    "[ERROR] Existen infinitos en X"
)

print("\n✓ X validada correctamente antes del split.")

# -----------------------------------------------------------------------------
# Train / Test Split
# -----------------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_train = X_train.copy()
X_test = X_test.copy()

# -----------------------------------------------------------------------------
# NUEVO: Funciones para Target Encoding Bayesiano.
# Cada encoding es una tasa específica, no el promedio de etiquetas 0/1/2.
# -----------------------------------------------------------------------------

def fit_smoothed_rate(categories, binary_target, smoothing=30):

    tabla = pd.DataFrame({
        "category": categories.astype(str).to_numpy(),
        "target": binary_target.astype(float).to_numpy()
    })

    global_rate = float(tabla["target"].mean())

    estadisticas = (
        tabla
        .groupby("category")["target"]
        .agg(["mean", "count"])
    )

    mapping = (
        (
            estadisticas["count"] * estadisticas["mean"] +
            smoothing * global_rate
        ) /
        (
            estadisticas["count"] + smoothing
        )
    )

    return mapping, global_rate


def apply_smoothed_rate(categories, mapping, global_rate):

    return (
        categories.astype(str)
        .map(mapping)
        .fillna(global_rate)
        .astype(float)
    )


def oof_smoothed_rate(
    categories,
    binary_target,
    smoothing=30,
    n_splits=5,
    seed=SEED
):

    encoded = pd.Series(
        index=categories.index,
        dtype=np.float64
    )

    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    for train_idx, valid_idx in splitter.split(categories, binary_target):

        categorias_fold_train = categories.iloc[train_idx]
        target_fold_train = binary_target.iloc[train_idx]

        categorias_fold_valid = categories.iloc[valid_idx]

        mapping_fold, global_fold = fit_smoothed_rate(
            categorias_fold_train,
            target_fold_train,
            smoothing=smoothing
        )

        encoded.iloc[valid_idx] = apply_smoothed_rate(
            categorias_fold_valid,
            mapping_fold,
            global_fold
        ).to_numpy()

    return encoded.astype(float)


# -----------------------------------------------------------------------------
# Recuperamos las rutas y aerolíneas de acuerdo con los índices del split.
# -----------------------------------------------------------------------------

train_routes = df_ml_pandas.loc[X_train.index, "route_id"]
test_routes = df_ml_pandas.loc[X_test.index, "route_id"]

train_carriers = df_ml_pandas.loc[X_train.index, "carrier_id"]
test_carriers = df_ml_pandas.loc[X_test.index, "carrier_id"]

# -----------------------------------------------------------------------------
# Tasa histórica de retraso por ruta y aerolínea.
# Clase 1 = retrasado.
# -----------------------------------------------------------------------------

train_is_delay = (y_train == 1).astype(int)

route_delay_mapping, route_delay_global = fit_smoothed_rate(
    train_routes,
    train_is_delay
)

carrier_delay_mapping, carrier_delay_global = fit_smoothed_rate(
    train_carriers,
    train_is_delay
)

X_train["route_hist_delay_rate"] = oof_smoothed_rate(
    train_routes,
    train_is_delay
)

X_test["route_hist_delay_rate"] = apply_smoothed_rate(
    test_routes,
    route_delay_mapping,
    route_delay_global
)

X_train["carrier_hist_delay_rate"] = oof_smoothed_rate(
    train_carriers,
    train_is_delay
)

X_test["carrier_hist_delay_rate"] = apply_smoothed_rate(
    test_carriers,
    carrier_delay_mapping,
    carrier_delay_global
)

# -----------------------------------------------------------------------------
# Tasa histórica de cancelación por ruta y aerolínea.
# Clase 2 = cancelado.
# -----------------------------------------------------------------------------

train_is_cancelled = (y_train == 2).astype(int)

route_cancel_mapping, route_cancel_global = fit_smoothed_rate(
    train_routes,
    train_is_cancelled
)

carrier_cancel_mapping, carrier_cancel_global = fit_smoothed_rate(
    train_carriers,
    train_is_cancelled
)

X_train["route_hist_cancel_rate"] = oof_smoothed_rate(
    train_routes,
    train_is_cancelled
)

X_test["route_hist_cancel_rate"] = apply_smoothed_rate(
    test_routes,
    route_cancel_mapping,
    route_cancel_global
)

X_train["carrier_hist_cancel_rate"] = oof_smoothed_rate(
    train_carriers,
    train_is_cancelled
)

X_test["carrier_hist_cancel_rate"] = apply_smoothed_rate(
    test_carriers,
    carrier_cancel_mapping,
    carrier_cancel_global
)

# -----------------------------------------------------------------------------
# Validación final después de target encoding.
# -----------------------------------------------------------------------------

assert not X_train.isnull().any().any(), (
    "[ERROR] Existen NaN en X_train después de target encoding"
)

assert not X_test.isnull().any().any(), (
    "[ERROR] Existen NaN en X_test después de target encoding"
)

assert not np.isinf(X_train.values).any(), (
    "[ERROR] Existen infinitos en X_train"
)

assert not np.isinf(X_test.values).any(), (
    "[ERROR] Existen infinitos en X_test"
)

# -----------------------------------------------------------------------------
# Resumen
# -----------------------------------------------------------------------------

FEATURE_COLS_FINAL = list(X_train.columns)

print("\n" + "=" * 70)
print("PARTICIÓN TRAIN / TEST (80/20)")
print("=" * 70)

print(f"\nNúmero FINAL de variables predictoras: {len(FEATURE_COLS_FINAL)}")

print("\nFeatures que entran a los modelos:")
for i, col in enumerate(FEATURE_COLS_FINAL, start=1):
    print(f"{i:02d}. {col}")

print(f"\n✓ X_train final: {X_train.shape}")
print(f"✓ X_test final: {X_test.shape}")

##Modelos

In [0]:
#----------------------------------------------------------------------------
# CELDA 6: ENTRENAMIENTO DEL RANDOM FOREST BASELINE
#---------------------------------------------------------------------------

print("_" * 70)
print("INICIALIZANDO Y ENTRENANDO RANDOM FOREST BASELINE")
print("_" * 70)

# 1. Instanciamos el modelo controlando rigurosamente la geometría de los árboles
rf_baseline = RandomForestClassifier(
    n_estimators=300,           # Cantidad de árboles para estabilizar el promedio
    max_depth=22,               # Profundidad límite para capturar interacciones sin memorizar
    min_samples_split= 15,       # Mínimo de muestras por nodo para autorizar una división
    min_samples_leaf=5,        # Agregamos este freno: cada hoja final debe tener al menos 30 vuelos.
    max_features="sqrt",        # Muestreo aleatorio de variables en cada división (log2/sqrt)
    random_state=SEED,          # Semilla global reproducible (42)
    class_weight="balanced",    # Balanceo de clases por inverso de proporciones
    n_jobs=-1                   # Paralelización masiva usando todos los cores del clúster
)

print(f"[INFO] Entrenando el bosque sobre {X_train.shape[0]:,} registros...")
tic = time.time()
rf_baseline.fit(X_train, y_train)
toc = time.time()
print(f"[SUCCESS] Entrenamiento completado en: {toc - tic:.2f} segundos.")

# -----------------------------------------------------------------------------
# PREDICCIONES EN AMBOS CONJUNTOS (DIAGNÓSTICO DE OVERFITTING)
# -----------------------------------------------------------------------------
print("\n[INFO] Ejecutando inferencias en Train y Test...")
y_pred_train = rf_baseline.predict(X_train)
y_pred_test = rf_baseline.predict(X_test)

# -----------------------------------------------------------------------------
# EVALUACIÓN DE MÉTRICAS MÁSTRICAS (MÉTODO CLASSIFICATION REPORT)
# -----------------------------------------------------------------------------
target_names = ["Clase 0: Puntual", "Clase 1: Retrasado", "Clase 2: Cancelado"]

print("\n" + "_" * 60)
print(" REPORT DE CLASIFICACIÓN — CONJUNTO DE ENTRENAMIENTO (TRAIN)")
print("_" * 60)
print(classification_report(y_train, y_pred_train, target_names=target_names))

print("\n" + "_" * 60)
print(" REPORT DE CLASIFICACIÓN — CONJUNTO DE PRUEBA FINAL (TEST)")
print("_" * 60)
print(classification_report(y_test, y_pred_test, target_names=target_names))

In [0]:
%pip install lightgbm

In [0]:
# __________________________________________________________________________
# - LIGHTGBM MULTICLASE
# - Se activa subsample correctamente con subsample_freq=1.
# - Se usa validación interna para early stopping.
# - No utiliza el conjunto test para ajustar el modelo.
# _________________________________________________________________________

import lightgbm as lgb
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

print("_" * 70)
print("INICIALIZANDO LIGHTGBM MULTICLASE")
print("_" * 70)

# -----------------------------------------------------------------------------
# Split interno para early stopping.
# El conjunto test final sigue reservado y no se usa para entrenar.
# -----------------------------------------------------------------------------

X_lgb_train, X_lgb_valid, y_lgb_train, y_lgb_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train
)

# -----------------------------------------------------------------------------
# Se conserva boost moderado para retrasados.
# -----------------------------------------------------------------------------

class_weights_per_class = {
    0: 1.0,
    1: 1.15,
    2: 1.0
}

sample_weights_lgb = np.array([
    class_weights_per_class[clase]
    for clase in y_lgb_train
])

# -----------------------------------------------------------------------------
# Modelo LightGBM.
# -----------------------------------------------------------------------------

lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,

    # Se permite un máximo alto, pero early stopping decide el número real.
    n_estimators=1800,
    learning_rate=0.035,
    num_leaves=63,
    min_child_samples=30,
    # subsample solo se activa cuando subsample_freq es mayor que cero.
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.3,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

print(f"Entrenando LightGBM con {X_lgb_train.shape[0]:,} vuelos...")
print(f"Validación interna: {X_lgb_valid.shape[0]:,} vuelos...")

lgbm_model.fit(
    X_lgb_train,
    y_lgb_train,
    sample_weight=sample_weights_lgb,

    eval_set=[
        (X_lgb_valid, y_lgb_valid)
    ],

    eval_metric="multi_logloss",

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=70,
            verbose=False
        ),
        lgb.log_evaluation(period=0)
    ]
)

print(
    f"\nMejor iteración encontrada: "
    f"{lgbm_model.best_iteration_}"
)

y_pred_lgb = lgbm_model.predict(X_test)

print("\n" + "_" * 70)
print("REPORT DE CLASIFICACIÓN LIGHTGBM — TEST FINAL")
print("_" * 70)

print(
    classification_report(
        y_test,
        y_pred_lgb,
        target_names=[
            "Clase 0: Puntual",
            "Clase 1: Retrasado",
            "Clase 2: Cancelado"
        ]
    )
)

In [0]:
# _____________________________________________________________________________
# REGRESIÓN LOGÍSTICA MULTINOMIAL (BENCHMARK)
# _________________________________________________________________________
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("_" * 70)
print("INICIALIZANDO Y ENTRENANDO REGRESIÓN LOGÍSTICA (BASELINE LINEAL)")
print("_" * 70)

# Instanciamos el modelo con hiperparámetros mejorados
lr_model = LogisticRegression(
    multi_class="multinomial",   # Softmax para multiclase
    solver="saga",               # SAGA soporta regularización y es eficiente para grandes datasets
    penalty="elasticnet",        # ElasticNet combina L1 y L2
    l1_ratio=0.3,                # Mezcla 30% L1, 70% L2
    C=2.0,                      # Menor regularización (C más alto)
    max_iter=1000,              # Más iteraciones para asegurar convergencia
    tol=1e-4,                   # Tolerancia más estricta
    random_state=SEED,
    n_jobs=-1                   # Paraleliza el cálculo de las clases
)

print(f"[INFO] Entrenando Regresión Logística sobre {X_train.shape[0]:,} registros...")
tic = time.time()
lr_model.fit(X_train, y_train)
toc = time.time()
print(f"[SUCCESS] Base lineal calculada en: {toc - tic:.2f} segundos.")

# Inferencias rápidas
y_pred_train_lr = lr_model.predict(X_train)
y_pred_test_lr = lr_model.predict(X_test)

target_names = ["Clase 0: Puntual", "Clase 1: Retrasado", "Clase 2: Cancelado"]

print("\n" + "_" * 60)
print(" REPORT DE CLASIFICACIÓN LOGISTIC REGRESSION — TEST FINAL")
print("_" * 60)
print(classification_report(y_test, y_pred_test_lr, target_names=target_names))

In [0]:
# ______________________________________________________________________
# MULTI-LAYER PERCEPTRON
#________________________________________________________________________________

import time

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

print("_" * 70)
print("INICIALIZANDO Y ENTRENANDO RED NEURONAL MLP")
print("_" * 70)

# -----------------------------------------------------------------------------
# StandardScaler normaliza todas las columnas antes de entrar al MLP.
# Esto es especialmente importante para Distance y CRSElapsedTime.
# -----------------------------------------------------------------------------

mlp_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(128, 64, 32),
                activation="relu",
                solver="adam",

                alpha=0.001,

                batch_size=256,
                max_iter=300,

                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=15,

                learning_rate_init=0.001,

                random_state=SEED
            )
        )
    ]
)

print(f"[INFO] Entrenando red neuronal sobre {X_train.shape[0]:,} registros...")

tic = time.time()

mlp_model.fit(
    X_train,
    y_train
)

toc = time.time()

print(
    f"[SUCCESS] Red neuronal entrenada en "
    f"{toc - tic:.2f} segundos."
)

y_pred_test_mlp = mlp_model.predict(X_test)

print("\n" + "_" * 70)
print("REPORT DE CLASIFICACIÓN MLP — TEST FINAL")
print("_" * 70)

print(
    classification_report(
        y_test,
        y_pred_test_mlp,
        target_names=[
            "Clase 0: Puntual",
            "Clase 1: Retrasado",
            "Clase 2: Cancelado"
        ]
    )
)

In [0]:
# __________________________________________________________________________
# PERSISTENCIA DE ARTEFACTOS ANALÍTICOS INTEGRAL
# __________________________________________________________________________

import os
import joblib
from datetime import datetime

print("=" * 70)
print("INICIANDO EXPORTACIÓN Y SALVAGUARDA DE TODO EL ECOSISTEMA")
print("=" * 70)

# -----------------------------------------------------------------------------
# 1. Verificación previa: confirma que los modelos fueron entrenados.
# -----------------------------------------------------------------------------

modelos_requeridos = {
    "lr_model": "Regresión Logística",
    "mlp_model": "Red Neuronal MLP",
    "rf_baseline": "Random Forest",
    "lgbm_model": "LightGBM"
}

modelos_faltantes = [
    nombre_variable
    for nombre_variable in modelos_requeridos
    if nombre_variable not in globals()
]

if len(modelos_faltantes) > 0:
    raise NameError(
        "Faltan modelos en memoria: "
        + ", ".join(modelos_faltantes)
        + ". Ejecutá las celdas de entrenamiento antes de exportar."
    )

# -----------------------------------------------------------------------------
# 2. Ruta persistente dentro del Volumen de Databricks.
#    No se guarda en el directorio temporal de la sesión.
# -----------------------------------------------------------------------------

ruta_modelos = "/Volumes/workspace/default/bronze_vuelos/modelos"
os.makedirs(ruta_modelos, exist_ok=True)

nombre_archivo = "ecosistema_completo_vuelos_v2.joblib"
ruta_archivo_final = os.path.join(ruta_modelos, nombre_archivo)

# -----------------------------------------------------------------------------
# 3. Empaquetamos modelos, metadata y transformaciones necesarias.
# -----------------------------------------------------------------------------

modelos_vuelos_hub = {
    "metadata": {
        "autor": "Cristhian Estuard Rojas Álvarez",
        "fecha_entrenamiento": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "version_modelo": "v2",
        "random_state": SEED,
        "clases_objetivo": {
            0: "Puntual",
            1: "Retrasado",
            2: "Cancelado"
        },
        "features_finales": FEATURE_COLS_FINAL,
        "cantidad_features": len(FEATURE_COLS_FINAL),
        "modelo_recomendado": "lightgbm",
        "metricas_test": {
            "random_forest_accuracy": 0.63,
            "lightgbm_accuracy": 0.64,
            "mlp_accuracy": 0.63,
            "logistic_regression_accuracy": 0.48
        }
    },

    "modelos": {
        "regresion_logistica": lr_model,
        "red_neuronal_mlp": mlp_model,


        # El nombre real del modelo Random Forest entrenado en el notebook.
        "random_forest": rf_baseline,

        "lightgbm": lgbm_model
    },

    # -------------------------------------------------------------------------
    # Se guardan los elementos usados para crear las features del modelo.
    # Son necesarios si luego se hacen predicciones sobre vuelos nuevos.
    # -------------------------------------------------------------------------
    "preprocesamiento": {
        "feature_columns_finales": FEATURE_COLS_FINAL,

        "columnas_one_hot_aerolineas": [
            columna
            for columna in FEATURE_COLS_FINAL
            if columna.startswith("carrier_")
        ],

        "route_delay_mapping": route_delay_mapping,
        "route_delay_global": route_delay_global,

        "carrier_delay_mapping": carrier_delay_mapping,
        "carrier_delay_global": carrier_delay_global,

        "route_cancel_mapping": route_cancel_mapping,
        "route_cancel_global": route_cancel_global,

        "carrier_cancel_mapping": carrier_cancel_mapping,
        "carrier_cancel_global": carrier_cancel_global
    }
}

# -----------------------------------------------------------------------------
# 4. Serialización y validación física.
# -----------------------------------------------------------------------------

print("\n[INFO] Serializando modelos y artefactos con joblib...")

try:
    joblib.dump(
        modelos_vuelos_hub,
        ruta_archivo_final,
        compress=3
    )

    tamano_mb = os.path.getsize(ruta_archivo_final) / (1024 * 1024)

    print("\n" + "=" * 70)
    print("RESPALDO MULTI-MODELO COMPLETADO CON ÉXITO!")
    print("=" * 70)

    print(f" Archivo maestro: {nombre_archivo}")
    print(f" Ruta persistente: {ruta_archivo_final}")
    print(f" Tamaño total: {tamano_mb:.2f} MB")

    print("\n✓ Modelos incluidos:")
    for i, modelo_nombre in enumerate(
        modelos_vuelos_hub["modelos"].keys(),
        start=1
    ):
        print(f"   {i}. {modelo_nombre}")

    print("\n Artefactos de preprocesamiento incluidos:")
    print("   - Features finales")
    print("   - One-hot encoding de aerolíneas")
    print("   - Historial de retrasos por ruta")
    print("   - Historial de cancelaciones por ruta")
    print("   - Historial de retrasos por aerolínea")
    print("   - Historial de cancelaciones por aerolínea")

    print("\n Modelo recomendado para despliegue: LightGBM")
    print(" Ecosistema listo para evaluación, comparación y despliegue técnico.")

except Exception as e:
    print(f"\n[ERROR] Falló la exportación: {str(e)}")